<a href="https://colab.research.google.com/github/brookdb/jwst-explorations/blob/main/02_europa_plumes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Europa's Plumes

## What This Notebook Does

This notebook retrieves real spectroscopic data from STScI's MAST
archive and looks for the spectral fingerprint of water vapor —
the specific wavelengths where water absorbs infrared light.

We'll go from raw telescope data to a scientific visualization,
explaining each step in terms of both the astronomy and the code.

---
*Data source: JWST/NIRSpec · MAST Archive · STScI/NASA*  
*Libraries: astropy · numpy · matplotlib · concurrent.futures*

In [38]:
!pip install astropy astroquery -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 42.4 MB/s eta 0:00:00


## Step 1 — Loading Our Tools

Before we can work with telescope data, we need to load our libraries.
Think of these as the instruments in a lab — each one has a specific job.

### The Astronomy Connection
JWST data is stored in FITS format (Flexible Image Transport System) —
a file standard that astronomy has used since 1981. Every telescope,
every mission, speaks this language. Astropy is the community-built
Python library that reads it.

### What Each Library Does
- **numpy** — represents our data as arrays of numbers. A spectrum
  is just an array of intensity values at different wavelengths.
- **matplotlib** — turns those arrays into visualizations
- **astropy.io.fits** — opens FITS files from any telescope
- **astropy.units** — keeps track of physical units (microns,
  Kelvin, etc.) so we don't mix up meters and miles
- **astropy.visualization** — tools for displaying astronomical
  data meaningfully
- **concurrent.futures** — lets Python do multiple things
  simultaneously. Important when working with large datasets.
- **requests / BytesIO** — fetches data from the internet and
  holds it in memory without saving to disk first

### Journal Note
*what do you already know about any of these?*
- I 've used the first two libraries and the and last one before but i've never used the astropy until now.
- I've seen multithreading in ML algorithms before as well.

*What feels unfamiliar?*
- I guess the fits file format is unfamiliar. Is it like a dictionary/JSON or more like a Java Object with classes and functions? How do we access items from it? or traverse it? or manuplate its elements?

In [39]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from astropy.io import fits
from astropy import units as u
from astropy.visualization import ZScaleInterval, ImageNormalize
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from io import BytesIO
import time

print("All libraries loaded.")
print(f"NumPy version: {np.__version__}")

All libraries loaded.
NumPy version: 2.0.2


## Step 2 — Understanding Our Data Source

### The Astronomy Connection: What is a Spectrum?

Last notebook we worked with an IMAGE — a 2D grid where each pixel
records how much light hit that spot on the detector.

This notebook works with a SPECTRUM — a 1D measurement of how much
light exists at each wavelength.

Imagine splitting light through a prism. A spectrum is the result —
intensity plotted against wavelength (color). Different molecules
absorb and emit light at specific wavelengths, like fingerprints.

Water vapor absorbs infrared light strongly around **2.7 micrometers**.
When JWST looked at Europa and saw a dip at that wavelength — that's
the fingerprint of water above the surface.

### The Data: FITS Format

A FITS file is structured like a filing cabinet:
- Each **extension** is a labeled drawer
- Each drawer contains a **header** (metadata as key/value pairs,
  like a dictionary) and **data** (a NumPy array)
- hdul['SCI'].data gives us the array
- hdul['SCI'].header['TELESCOP'] gives us a header value

### The Modeling Question
We are asking: at what wavelengths does Europa's brightness drop?
A drop at 2.7 micrometers = water vapor absorbing that light.
That absorption dip IS the detection of water.

### Journal Note
*The FITS format now makes sense to me as a dictionary. What surprised me about this is how it looks when I opened the file in my editor. Initially, I thought it might be just a clean file that you can open and look at using a code editor (like a JSON file). However, when I did that, I noticed it contains a lot of weird symbols (?@��ưt          ) alongside text that looks like it's a hybrid of markdown and YAML syntax. It kinda reminds me of a database file.*

In [40]:
# Europa NIRSpec observation from MAST
# Program ID: 1250 - JWST Europa observation
# NIRSpec IFU - Integral Field Unit spectroscopy
# This gives us a CUBE: 2D spatial + 1D spectral

url = "https://mast.stsci.edu/api/v0.1/Download/file?uri=mast:JWST/product/jw01250006001_03101_00001_nrs1_x1d.fits"

print("Fetching Europa spectral data from MAST...")
print("This is real NIRSpec data from JWST Program 1250")
print("The same dataset used in the 2023 Science paper detecting water vapor\n")

start = time.time()
response = requests.get(url, stream=True)
elapsed = time.time() - start

print(f"Response status: {response.status_code}")
print(f"Content type: {response.headers.get('content-type')}")
print(f"Download time: {elapsed:.2f} seconds")
print(f"File size: {len(response.content) / 1024:.1f} KB")

Fetching Europa spectral data from MAST...
This is real NIRSpec data from JWST Program 1250
The same dataset used in the 2023 Science paper detecting water vapor

Response status: 200
Content type: application/octet-stream
Download time: 0.36 seconds
File size: 675.0 KB


## Step 3 — Exploring the File Structure

### The Astronomy Connection: What is NIRSpec x1d?

NIRSpec is JWST's spectrograph — it splits light by wavelength.
"x1d" means "extracted 1D spectrum" — the pipeline has already
processed the raw detector data into a clean wavelength vs.
flux table.

Flux means "how much light per unit area per unit time" — it's
the astronomer's word for brightness at a specific wavelength.

### The Data Structure Question
Before we visualize anything, we need to understand what's
actually in this file. Good scientists (and good engineers)
always explore their data before analyzing it.

This is like running console.log() before you build the UI —
you need to know the shape of your data first.

### Journal Note
*What I noticed about the raw FITS file in my editor: it looked
like db file because the weird symbols are binary encoded numerical data. Now I understand that the binary blob is numerical data from the image or spectrum stored as raw bytes and the readable portions are the headers stored as plain ASCII. So you need Astropy to interprate the blob. I'm guessing the data is encoded for machines, not humans.*

*Looking at the columns, I understand what flux is (like magnetic flux, dot product of area and field vectors) and how it fits in this model. Not sure how surface brightness and background contribute to our model. Also not sure why we have Jy (I'm guessing jules?) for flux and MJy/sr for the other two. I'm also curios about the error calculations both from a statistical and astronomical perspectives.*

In [41]:
with fits.open(BytesIO(response.content)) as hdul:
    # First — always look at the overall structure
    # This is like console.log(Object.keys(data)) in JavaScript
    print("=== FILE STRUCTURE ===")
    hdul.info()

    # Now look inside the first science extension
    # x1d files store spectra as binary tables, not images
    # Each row is a different spectral order or detector segment
    print("\n=== EXTENSION 1 COLUMNS ===")
    print("(These are the 'keys' of our spectral data)")
    cols = hdul[1].columns
    for col in cols:
        print(f"  {col.name:20s} — {col.unit or 'no unit'}")

    # Extract the actual data
    # WAVELENGTH tells us WHERE in the spectrum we are
    # FLUX tells us HOW BRIGHT Europa is at that wavelength
    # ERROR tells us HOW CERTAIN we are of that measurement
    spec_data = hdul[1].data
    wavelength = spec_data['WAVELENGTH']  # in microns
    flux = spec_data['FLUX']              # in MJy/sr
    error = spec_data['FLUX_ERROR']            # uncertainty

    print(f"\n=== DATA SHAPE ===")
    print(f"Wavelength array: {wavelength.shape} points")
    print(f"Wavelength range: {wavelength.min():.3f} to {wavelength.max():.3f} microns")
    print(f"This covers: near-infrared light invisible to human eyes")
    print(f"\nWater vapor fingerprint at: ~2.7 microns")
    print(f"Is 2.7 microns in our range? {wavelength.min():.2f} < 2.7 < {wavelength.max():.2f}: "
          f"{wavelength.min() < 2.7 < wavelength.max()}")

=== FILE STRUCTURE ===
Filename: <class '_io.BytesIO'>
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     257   ()      
  1  EXTRACT1D     1 BinTableHDU     70   3915R x 18C   [D, D, D, D, D, D, D, D, D, D, D, J, D, D, D, D, D, D]   
  2  ASDF          1 BinTableHDU     11   1R x 1C   [107276B]   

=== EXTENSION 1 COLUMNS ===
(These are the 'keys' of our spectral data)
  WAVELENGTH           — um
  FLUX                 — Jy
  FLUX_ERROR           — Jy
  FLUX_VAR_POISSON     — Jy^2
  FLUX_VAR_RNOISE      — Jy^2
  FLUX_VAR_FLAT        — Jy^2
  SURF_BRIGHT          — MJy/sr
  SB_ERROR             — MJy/sr
  SB_VAR_POISSON       — (MJy/sr)^2
  SB_VAR_RNOISE        — (MJy/sr)^2
  SB_VAR_FLAT          — (MJy/sr)^2
  DQ                   — no unit
  BACKGROUND           — MJy/sr
  BKGD_ERROR           — MJy/sr
  BKGD_VAR_POISSON     — (MJy/sr)^2
  BKGD_VAR_RNOISE      — (MJy/sr)^2
  BKGD_VAR_FLAT        — (MJy/sr)^2
  NPIXELS              — no u

## Step 4 — Understanding Our Columns

### Units Matter — The Astronomy Connection

**FLUX (Jy)** — Jansky, named after Karl Jansky (1932). Measures
energy arriving at the telescope per unit frequency. Used for
point sources like stars.

**SURF_BRIGHT (MJy/sr)** — Surface brightness per steradian
(solid angle unit). Used for extended objects. Europa is a
disk in JWST's view, not a point — so this is our physically
meaningful measurement.

**BACKGROUND (MJy/sr)** — Everything that isn't Europa:
zodiacal dust, telescope thermal emission, Jupiter's scattered
light. Must be subtracted to see Europa's true spectrum.
This is like removing DC offset from a signal.

**The Three Error Sources — Error Propagation in Space**

Total_Error = √(VAR_POISSON + VAR_RNOISE + VAR_FLAT)

- VAR_POISSON: photon noise — light is quantized, photons  
  arrive randomly. Fundamental quantum physics.
- VAR_RNOISE: read noise — detector electronics uncertainty.  
  Constant regardless of signal strength.
- VAR_FLAT: flat field calibration uncertainty — the detector  
  isn't perfectly uniform.

This is the same error propagation from physics lab —
independent uncertainties add in quadrature.

### The Model We're Building
Signal = SURF_BRIGHT - BACKGROUND ± SB_ERROR
A dip in that signal at 2.7 microns = water vapor absorbing
infrared light = Europa's ocean reaching the surface.

### Journal Note
*The unit that surprised me most was surface brightness because I never did measurments of disk like objects (I've done spectral measurments of a sun spot at Mt. Wilson but that still is looking at a point source) so the units where not making sense at first.*

*When I pointed at the sunspot, I was narrowing my field of view so tightly that the sunspot occupied essentially one point on the detector. All the light came from one location. That's why FLUX works — I'm measuring total energy arriving, divided by the detector area.*

*Europa is different because JWST can actually resolve it as a disk — multiple detector pixels each capture a different patch of Europa's surface. If I used FLUX I'd get a number that depends on how big Europa looks in the telescope, which changes with distance. That's not a physical property of Europa, it's an accident of geometry.*

*Surface brightness solves this by normalizing by the solid angle — the angular size of each patch of sky each pixel sees. Now the number is intrinsic to Europa itself, independent of distance or telescope size. That's why astronomers prefer it for extended objects.*

*The steradian is just the unit of that solid angle. Your detector is divided into pixels, each pixel sees a certain patch of sky measured in steradians, and surface brightness tells you how bright that patch is. The grid on Europa in the diagram shows exactly that — each cell is one unit of sky angle.*

In [42]:
animation1 = """

<style>
  body { margin:0; background:#05080f; display:flex; align-items:center; justify-content:center; min-height:300px; }
  canvas { display:block; }
</style>
<canvas id="c" width="680" height="280"></canvas>
<script>
const canvas = document.getElementById('c');
const ctx = canvas.getContext('2d');
let phase = 0, t = 0;
const W = 680, H = 280;
const starX = 110, starY = 140;
const detX = 510, detY = 140;

// Flat-top hexagon: vertices at angles 0,60,120,180,240,300
// gives flat horizontal edges top and bottom
function hexPoints(cx, cy, r) {
  const pts = [];
  for (let i = 0; i < 6; i++) {
    const a = Math.PI / 3 * i;
    pts.push([cx + r * Math.cos(a), cy + r * Math.sin(a)]);
  }
  return pts;
}

function drawHex(cx, cy, r, fill, stroke, sw) {
  const pts = hexPoints(cx, cy, r);
  ctx.beginPath();
  ctx.moveTo(pts[0][0], pts[0][1]);
  for (let i = 1; i < 6; i++) ctx.lineTo(pts[i][0], pts[i][1]);
  ctx.closePath();
  if (fill)   { ctx.fillStyle   = fill;   ctx.fill(); }
  if (stroke) { ctx.strokeStyle = stroke; ctx.lineWidth = sw || 0.5; ctx.stroke(); }
}

// JWST honeycomb: flat-top hexagons, R=13
// For flat-top: colSpacing = R*1.5, rowSpacing = R*sqrt(3), stagger = R*sqrt(3)/2
function buildJWST(cx, cy) {
  const R = 13;
  const cs = R * 1.5;                  // col spacing  = 19.5
  const rs = R * Math.sqrt(3);         // row spacing  = 22.52
  const st = R * Math.sqrt(3) / 2;     // stagger      = 11.26

  // 3-4-5-4-3 columns: x offsets -2cs, -cs, 0, cs, 2cs
  // Col y positions:
  // Count 3 (even cols): -rs, 0, rs
  // Count 4 (odd cols):  -3st, -st, st, 3st  (= -rs*1.5, -rs*0.5, rs*0.5, rs*1.5)
  // Count 5 (center):    -2rs, -rs, 0, rs, 2rs
  const colDefs = [
    { dx: -2*cs, ys: [-rs, 0, rs] },
    { dx: -cs,   ys: [-3*st, -st, st, 3*st] },
    { dx: 0,     ys: [-2*rs, -rs, rs, 2*rs] },
    { dx: cs,    ys: [-3*st, -st, st, 3*st] },
    { dx: 2*cs,  ys: [-rs, 0, rs] },
  ];

  const positions = [];
  colDefs.forEach(col => {
    col.ys.forEach(dy => {
      positions.push([cx + col.dx, cy + dy]);
    });
  });
  return { positions, R };
}

const { positions: hexes, R: HR } = buildJWST(detX, detY);

const bgStars = [
  [55,35,0.9,0.4],[140,70,0.7,0.3],[80,190,1.0,0.5],
  [35,245,0.6,0.3],[185,25,0.8,0.4],[630,45,0.9,0.3],
  [655,185,0.7,0.4],[610,250,1.0,0.3]
];

const rays = [
  { dy:  0,  amp: 9, alpha: 1.0,  lw: 1.2 },
  { dy: -12, amp: 7, alpha: 0.75, lw: 0.9 },
  { dy:  12, amp: 7, alpha: 0.75, lw: 0.9 },
  { dy: -22, amp: 5, alpha: 0.45, lw: 0.6 },
  { dy:  22, amp: 5, alpha: 0.45, lw: 0.6 },
];

function drawWave(yCenter, amp, alpha, lw) {
  const x0 = starX + 12, x1 = detX - 58;
  const wl = 38;
  ctx.beginPath();
  ctx.strokeStyle = `rgba(255,251,230,${alpha})`;
  ctx.lineWidth = lw;
  ctx.lineJoin = 'round';
  let first = true;
  for (let x = x0; x <= x1; x += 1.5) {
    const y = yCenter + amp * Math.sin((x - x0) / wl * Math.PI * 2 - phase);
    if (first) { ctx.moveTo(x, y); first = false; }
    else ctx.lineTo(x, y);
  }
  ctx.stroke();
}

function drawStar(pulse) {
  const g = ctx.createRadialGradient(starX, starY, 0, starX, starY, 44);
  g.addColorStop(0, `rgba(255,251,230,${0.9+pulse*0.1})`);
  g.addColorStop(0.4, `rgba(250,199,117,${0.5+pulse*0.1})`);
  g.addColorStop(1, 'rgba(250,199,117,0)');
  ctx.beginPath(); ctx.arc(starX, starY, 44, 0, Math.PI*2);
  ctx.fillStyle = g; ctx.fill();

  const sa = 0.4 + pulse * 0.6;
  ctx.strokeStyle = `rgba(255,251,230,${sa})`; ctx.lineWidth = 0.8;
  [[starX,starY-13,starX,starY-32],[starX,starY+13,starX,starY+32],
   [starX-13,starY,starX-32,starY],[starX+13,starY,starX+32,starY]].forEach(([x1,y1,x2,y2])=>{
    ctx.beginPath(); ctx.moveTo(x1,y1); ctx.lineTo(x2,y2); ctx.stroke();
  });

  const c = ctx.createRadialGradient(starX, starY, 0, starX, starY, 10);
  c.addColorStop(0,'#ffffff'); c.addColorStop(0.6,'#fffbe6'); c.addColorStop(1,'#FAC775');
  ctx.beginPath(); ctx.arc(starX, starY, 9, 0, Math.PI*2);
  ctx.fillStyle = c; ctx.fill();
}

function drawDetector(pulse) {
  const gp = (pulse + 1) / 2;

  hexes.forEach(([hx, hy]) => {
    const isCenter = Math.abs(hx - detX) < 1 && Math.abs(hy - detY) < 1;
    if (isCenter) {
      // halo
      const h = ctx.createRadialGradient(detX, detY, 0, detX, detY, 22);
      h.addColorStop(0, `rgba(250,199,117,${0.65*gp})`);
      h.addColorStop(1, 'rgba(250,199,117,0)');
      ctx.beginPath(); ctx.arc(detX, detY, 22, 0, Math.PI*2);
      ctx.fillStyle = h; ctx.fill();
      drawHex(hx, hy, HR, `rgba(250,199,117,${0.5+0.5*gp})`, '#EF9F27', 0.8);
    } else {
      drawHex(hx, hy, HR, '#B8860B', '#3a2800', 0.5);
    }
  });
}

function frame() {
  ctx.clearRect(0, 0, W, H);
  ctx.fillStyle = '#05080f'; ctx.fillRect(0, 0, W, H);

  bgStars.forEach(([x,y,r,a]) => {
    ctx.beginPath(); ctx.arc(x,y,r,0,Math.PI*2);
    ctx.fillStyle = `rgba(255,255,255,${a})`; ctx.fill();
  });

  t += 0.03; phase += 0.12;
  const pulse = Math.sin(t);

  drawStar(pulse);
  rays.forEach(r => drawWave(starY + r.dy, r.amp, r.alpha, r.lw));
  drawDetector(pulse);

  ctx.fillStyle = '#666688'; ctx.font = '11px Arial'; ctx.textAlign = 'center';
  ctx.fillText('Point source', starX, 215);
  ctx.fillText('1 segment lit', detX, 252);

  requestAnimationFrame(frame);
}
frame();
</script>
"""
with open('animation_01_point_source.html', 'w') as f:
    f.write(animation1)

print("Saved.")


Saved.


In [43]:
from IPython.display import HTML
HTML(animation1)

In [44]:
animation2 = """
<style>
  body { margin:0; background:#05080f; display:flex; align-items:center; justify-content:center; min-height:300px; }
  canvas { display:block; }
</style>
<canvas id="c" width="680" height="280"></canvas>
<script>
const canvas = document.getElementById('c');
const ctx = canvas.getContext('2d');
let phase = 0, t = 0;
const W = 680, H = 280;
const detX = 510, detY = 140;
const europaX = 110, europaY = 140, europaR = 55;

function hexPoints(cx, cy, r) {
  const pts = [];
  for (let i = 0; i < 6; i++) {
    const a = Math.PI / 3 * i;
    pts.push([cx + r * Math.cos(a), cy + r * Math.sin(a)]);
  }
  return pts;
}

function drawHex(cx, cy, r, fill, stroke, sw) {
  const pts = hexPoints(cx, cy, r);
  ctx.beginPath();
  ctx.moveTo(pts[0][0], pts[0][1]);
  for (let i = 1; i < 6; i++) ctx.lineTo(pts[i][0], pts[i][1]);
  ctx.closePath();
  if (fill)   { ctx.fillStyle = fill;   ctx.fill(); }
  if (stroke) { ctx.strokeStyle = stroke; ctx.lineWidth = sw||0.5; ctx.stroke(); }
}

function buildJWST(cx, cy) {
  const R = 13;
  const cs = R * 1.5;
  const rs = R * Math.sqrt(3);
  const st = R * Math.sqrt(3) / 2;
  const colDefs = [
    { dx: -2*cs, ys: [-rs, 0, rs] },
    { dx: -cs,   ys: [-3*st, -st, st, 3*st] },
    { dx: 0,     ys: [-2*rs, -rs, 0, rs, 2*rs] },
    { dx: cs,    ys: [-3*st, -st, st, 3*st] },
    { dx: 2*cs,  ys: [-rs, 0, rs] },
  ];
  const positions = [];
  colDefs.forEach(col => col.ys.forEach(dy => positions.push([cx+col.dx, cy+dy])));
  return { positions, R };
}

const { positions: hexes, R: HR } = buildJWST(detX, detY);

const bgStars = [
  [55,35,0.9,0.4],[620,45,0.9,0.3],[655,185,0.7,0.4],
  [610,250,1.0,0.3],[35,245,0.6,0.3],[185,25,0.8,0.4]
];

// One wave source per hex segment — each originates from a different point on Europa
// Europa has 19 hex segments, so 19 source points spread across the disk
function getEuropaSourcePoints() {
  // Distribute source points across Europa's disk matching hex count
  const pts = [];
  // Center
  pts.push([europaX, europaY]);
  // Inner ring — 6 points
  for (let i = 0; i < 6; i++) {
    const a = Math.PI / 3 * i;
    pts.push([europaX + europaR * 0.4 * Math.cos(a), europaY + europaR * 0.4 * Math.sin(a)]);
  }
  // Outer ring — 12 points
  for (let i = 0; i < 12; i++) {
    const a = Math.PI / 6 * i;
    pts.push([europaX + europaR * 0.78 * Math.cos(a), europaY + europaR * 0.78 * Math.sin(a)]);
  }
  return pts;
}

const sourcePts = getEuropaSourcePoints();

function drawWave(x0, y0, x1, y1, phaseOffset, alpha, lw) {
  const dx = x1 - x0, dy = y1 - y0;
  const len = Math.sqrt(dx*dx + dy*dy);
  const steps = Math.ceil(len / 1.5);
  const wl = 36;
  // Perpendicular direction for sine displacement
  const px = -dy / len, py = dx / len;

  ctx.beginPath();
  ctx.strokeStyle = `rgba(180,210,255,${alpha})`;
  ctx.lineWidth = lw;
  ctx.lineJoin = 'round';
  for (let i = 0; i <= steps; i++) {
    const frac = i / steps;
    const bx = x0 + dx * frac;
    const by = y0 + dy * frac;
    const amp = 5 * Math.sin(Math.PI * frac); // taper at ends
    const sine = amp * Math.sin(frac * len / wl * Math.PI * 2 - phase + phaseOffset);
    const wx = bx + px * sine;
    const wy = by + py * sine;
    if (i === 0) ctx.moveTo(wx, wy);
    else ctx.lineTo(wx, wy);
  }
  ctx.stroke();
}

function drawEuropa() {
  // Base icy sphere
  const grad = ctx.createRadialGradient(
    europaX - europaR*0.3, europaY - europaR*0.3, europaR*0.1,
    europaX, europaY, europaR
  );
  grad.addColorStop(0,   '#e8f4ff');
  grad.addColorStop(0.3, '#c5dff5');
  grad.addColorStop(0.7, '#8ab8e8');
  grad.addColorStop(1,   '#4a7fb5');
  ctx.beginPath();
  ctx.arc(europaX, europaY, europaR, 0, Math.PI*2);
  ctx.fillStyle = grad;
  ctx.fill();

  // Surface crack lines (characteristic Europa feature)
  ctx.strokeStyle = 'rgba(80,120,180,0.6)';
  ctx.lineWidth = 0.8;
  // A few reddish-brown lineae
  const cracks = [
    [[70,110],[130,160],[150,175]],
    [[85,175],[120,145],[160,130]],
    [[95,115],[140,135],[160,120]],
    [[75,150],[115,125],[145,115]],
    [[100,170],[135,155],[155,145]],
  ];
  cracks.forEach(pts => {
    ctx.beginPath();
    ctx.strokeStyle = 'rgba(120,80,60,0.45)';
    ctx.lineWidth = 0.9;
    ctx.moveTo(pts[0][0], pts[0][1]);
    for (let i = 1; i < pts.length; i++) ctx.lineTo(pts[i][0], pts[i][1]);
    ctx.stroke();
  });

  // Subtle terminator shadow on right edge
  const shadow = ctx.createRadialGradient(
    europaX + europaR*0.5, europaY, europaR*0.2,
    europaX, europaY, europaR
  );
  shadow.addColorStop(0.6, 'rgba(0,10,30,0)');
  shadow.addColorStop(1,   'rgba(0,10,30,0.45)');
  ctx.beginPath();
  ctx.arc(europaX, europaY, europaR, 0, Math.PI*2);
  ctx.fillStyle = shadow;
  ctx.fill();

  // Thin atmosphere glow
  const atmo = ctx.createRadialGradient(europaX, europaY, europaR*0.9, europaX, europaY, europaR*1.15);
  atmo.addColorStop(0,   'rgba(150,200,255,0.15)');
  atmo.addColorStop(1,   'rgba(150,200,255,0)');
  ctx.beginPath();
  ctx.arc(europaX, europaY, europaR*1.15, 0, Math.PI*2);
  ctx.fillStyle = atmo;
  ctx.fill();

  // Jupiter barely visible in background (large soft circle, very faint)
  const jup = ctx.createRadialGradient(30, 80, 10, 30, 80, 90);
  jup.addColorStop(0,   'rgba(200,160,100,0.08)');
  jup.addColorStop(0.6, 'rgba(180,140,80,0.04)');
  jup.addColorStop(1,   'rgba(180,140,80,0)');
  ctx.beginPath();
  ctx.arc(30, 80, 90, 0, Math.PI*2);
  ctx.fillStyle = jup;
  ctx.fill();
}

function drawDetector(pulse) {
  const gp = (pulse + 1) / 2;
  hexes.forEach(([hx, hy], i) => {
    // Each hex glows with slight phase variation for visual richness
    const localGlow = (Math.sin(t + i * 0.3) + 1) / 2;
    const brightness = 0.4 + 0.6 * localGlow;
    drawHex(hx, hy, HR, `rgba(180,210,255,${0.15 + 0.25*brightness})`, '#6B4F00', 0.5);
    // Gold mirror tint
    drawHex(hx, hy, HR, null, `rgba(184,134,11,${0.6 + 0.3*brightness})`, 0.5);
  });
}

function frame() {
  ctx.clearRect(0, 0, W, H);
  ctx.fillStyle = '#05080f'; ctx.fillRect(0, 0, W, H);

  bgStars.forEach(([x,y,r,a]) => {
    ctx.beginPath(); ctx.arc(x,y,r,0,Math.PI*2);
    ctx.fillStyle = `rgba(255,255,255,${a})`; ctx.fill();
  });

  t += 0.025;
  phase += 0.11;
  const pulse = Math.sin(t);

  drawEuropa();

  // Draw waves from each source point to corresponding hex segment
  hexes.forEach(([hx, hy], i) => {
    const src = sourcePts[i] || sourcePts[sourcePts.length - 1];
    const dist = Math.sqrt((hx-src[0])**2 + (hy-src[1])**2);
    const alpha = 0.35 + 0.2 * Math.sin(t + i * 0.4);
    drawWave(src[0], src[1], hx, hy, i * 0.33, alpha, 0.7);
  });

  drawDetector(pulse);

  ctx.fillStyle = '#666688'; ctx.font = '11px Arial'; ctx.textAlign = 'center';
  ctx.fillText('Extended source', europaX, 215);
  ctx.fillText('All segments lit', detX, 252);

  requestAnimationFrame(frame);
}
frame();
</script>

"""
with open('animation_02_extended_source.html', 'w') as f:
    f.write(animation2)

print("Saved.")


Saved.


In [45]:
from IPython.display import HTML
HTML(animation2)

In [46]:
animation3 = """

<style>
  body { margin:0; background:#05080f; display:flex; align-items:center; justify-content:center; min-height:320px; }
  canvas { display:block; }
</style>
<canvas id="c" width="680" height="300"></canvas>
<script>
const canvas = document.getElementById('c');
const ctx = canvas.getContext('2d');
const W = 680, H = 300;
const hexCX = 570, hexCY = 150, hexR = 18;
const FIXED_PATCH_R = 14;
const EUROPA_CY = 150;

// State machine
const PAUSE_FRAMES = 144;
const MOVE_FRAMES  = 110;
let stateFrame = 0;
let state = 'MOVING_IN';
let cycle = 0;

// Spring physics for SB bar
let sbSpringPos = 90;   // current width
let sbSpringVel = 0;
const SB_REST   = 90;   // resting width
const SB_SPRING = 0.18; // spring constant
const SB_DAMP   = 0.72; // damping

let prevCycle = 0;

function easeInOut(t) {
  return t < 0.5 ? 2*t*t : -1+(4-2*t)*t;
}

function hexPoints(cx, cy, r) {
  const pts = [];
  for (let i = 0; i < 6; i++) {
    const a = Math.PI/3*i;
    pts.push([cx + r*Math.cos(a), cy + r*Math.sin(a)]);
  }
  return pts;
}

function drawHex(cx, cy, r, fill, stroke, sw) {
  const pts = hexPoints(cx, cy, r);
  ctx.beginPath();
  ctx.moveTo(pts[0][0], pts[0][1]);
  for (let i=1;i<6;i++) ctx.lineTo(pts[i][0], pts[i][1]);
  ctx.closePath();
  if (fill)   { ctx.fillStyle=fill;   ctx.fill(); }
  if (stroke) { ctx.strokeStyle=stroke; ctx.lineWidth=sw||0.5; ctx.stroke(); }
}

function drawEuropa(ex, ey, er, brightness) {
  // brightness: 0=far(dim) 1=close(bright) — applied to color intensity
  const b = Math.max(0.15, brightness); // minimum visibility even when far

  const grad = ctx.createRadialGradient(ex-er*0.3, ey-er*0.3, er*0.1, ex, ey, er);
  grad.addColorStop(0,   `rgba(${Math.round(180+75*b)},${Math.round(215+40*b)},255,1)`);
  grad.addColorStop(0.3, `rgba(${Math.round(120+75*b)},${Math.round(170+55*b)},${Math.round(205+50*b)},1)`);
  grad.addColorStop(0.7, `rgba(${Math.round(55+85*b)},${Math.round(115+69*b)},${Math.round(155+73*b)},1)`);
  grad.addColorStop(1,   `rgba(${Math.round(20+54*b)},${Math.round(65+62*b)},${Math.round(105+74*b)},1)`);
  ctx.beginPath(); ctx.arc(ex, ey, er, 0, Math.PI*2);
  ctx.fillStyle = grad; ctx.fill();

  ctx.save();
  ctx.beginPath(); ctx.arc(ex, ey, er, 0, Math.PI*2); ctx.clip();
  ctx.strokeStyle = `rgba(120,80,60,${0.2+b*0.45})`;
  ctx.lineWidth = Math.max(0.4, er/35);
  [
    [[-0.4,-0.2],[0.1,0.35]],
    [[-0.2,0.4],[0.4,-0.1]],
    [[-0.3,-0.4],[0.3,-0.1]],
    [[0.2,-0.35],[0.45,0.2]],
  ].forEach(([a,bb]) => {
    ctx.beginPath();
    ctx.moveTo(ex+er*a[0], ey+er*a[1]);
    ctx.lineTo(ex+er*bb[0], ey+er*bb[1]);
    ctx.stroke();
  });
  ctx.restore();

  // Atmosphere — brighter when close
  const atmo = ctx.createRadialGradient(ex,ey,er*0.85,ex,ey,er*1.25);
  atmo.addColorStop(0, `rgba(150,200,255,${0.04+b*0.18})`);
  atmo.addColorStop(1, 'rgba(150,200,255,0)');
  ctx.beginPath(); ctx.arc(ex,ey,er*1.25,0,Math.PI*2);
  ctx.fillStyle=atmo; ctx.fill();
}

function drawFixedPatch(ex, ey, brightness) {
  const b = Math.max(0.2, brightness);
  const pg = ctx.createRadialGradient(ex, ey, 0, ex, ey, FIXED_PATCH_R+2);
  pg.addColorStop(0,   `rgba(255,220,80,${0.4+b*0.6})`);
  pg.addColorStop(0.7, `rgba(255,200,60,${0.15+b*0.3})`);
  pg.addColorStop(1,   'rgba(255,200,60,0)');
  ctx.beginPath(); ctx.arc(ex, ey, FIXED_PATCH_R, 0, Math.PI*2);
  ctx.fillStyle = pg; ctx.fill();
  ctx.strokeStyle = `rgba(255,220,80,${0.6+b*0.4})`;
  ctx.lineWidth = 1.2;
  ctx.stroke();
}

function drawSightLines(ex, ey) {
  ctx.strokeStyle = 'rgba(255,220,80,0.28)';
  ctx.lineWidth = 0.7;
  ctx.setLineDash([4,3]);
  ctx.beginPath();
  ctx.moveTo(hexCX - hexR, hexCY);
  ctx.lineTo(ex, ey - FIXED_PATCH_R);
  ctx.stroke();
  ctx.beginPath();
  ctx.moveTo(hexCX - hexR, hexCY);
  ctx.lineTo(ex, ey + FIXED_PATCH_R);
  ctx.stroke();
  ctx.setLineDash([]);
}

function drawDetector(brightness) {
  const b = brightness; // 0 to 1

  // Very large outer bloom when bright
  const bloom = ctx.createRadialGradient(hexCX, hexCY, 0, hexCX, hexCY, hexR*5);
  bloom.addColorStop(0,   `rgba(255,180,20,${b*0.55})`);
  bloom.addColorStop(0.35,`rgba(255,140,0,${b*0.35})`);
  bloom.addColorStop(0.7, `rgba(255,100,0,${b*0.12})`);
  bloom.addColorStop(1,   'rgba(255,80,0,0)');
  ctx.beginPath(); ctx.arc(hexCX, hexCY, hexR*5, 0, Math.PI*2);
  ctx.fillStyle=bloom; ctx.fill();

  // Tight inner glow
  const ig = ctx.createRadialGradient(hexCX, hexCY, 0, hexCX, hexCY, hexR*2);
  ig.addColorStop(0,   `rgba(255,255,200,${0.1+b*0.9})`);
  ig.addColorStop(0.5, `rgba(255,220,80,${0.05+b*0.7})`);
  ig.addColorStop(1,   'rgba(255,180,20,0)');
  ctx.beginPath(); ctx.arc(hexCX, hexCY, hexR*2, 0, Math.PI*2);
  ctx.fillStyle=ig; ctx.fill();

  // Hex face — near-white when blazing, dark gold when dim
  const fr = 255;
  const fg = Math.round(120 + b*135);
  const fb = Math.round(10  + b*220);
  const fa = 0.1 + b*0.9;
  const sr = 255;
  const sg = Math.round(150 + b*105);
  const sb2= Math.round(20  + b*200);
  drawHex(hexCX, hexCY, hexR,
    `rgba(${fr},${fg},${fb},${fa})`,
    `rgba(${sr},${sg},${sb2},${0.5+b*0.5})`,
    1.8
  );
}

const bgStars = [
  [30,25,0.8,0.3],[100,15,0.7,0.4],[300,20,0.8,0.3],
  [20,270,0.7,0.3],[200,285,0.8,0.4],[400,10,0.6,0.3]
];

function frame() {
  ctx.clearRect(0,0,W,H);
  ctx.fillStyle='#05080f'; ctx.fillRect(0,0,W,H);

  bgStars.forEach(([x,y,r,a]) => {
    ctx.beginPath(); ctx.arc(x,y,r,0,Math.PI*2);
    ctx.fillStyle=`rgba(255,255,255,${a})`; ctx.fill();
  });

  // State machine
  stateFrame++;
  if (state === 'MOVING_IN') {
    cycle = easeInOut(Math.min(1, stateFrame / MOVE_FRAMES));
    if (stateFrame >= MOVE_FRAMES) { state = 'PAUSE_CLOSE'; stateFrame = 0; }
  } else if (state === 'PAUSE_CLOSE') {
    cycle = 1;
    if (stateFrame >= PAUSE_FRAMES) { state = 'MOVING_OUT'; stateFrame = 0; }
  } else if (state === 'MOVING_OUT') {
    cycle = 1 - easeInOut(Math.min(1, stateFrame / MOVE_FRAMES));
    if (stateFrame >= MOVE_FRAMES) { state = 'PAUSE_FAR'; stateFrame = 0; }
  } else if (state === 'PAUSE_FAR') {
    cycle = 0;
    if (stateFrame >= PAUSE_FRAMES) { state = 'MOVING_IN'; stateFrame = 0; }
  }

  // Europa params
  const europaX = 110 + cycle * 150;
  const europaY = EUROPA_CY;
  const europaR  = 18 + cycle * 40;

  // Brightness: inverse square, 0=far 1=close, ALWAYS positive and visible
  const distMax = 150;
  const brightness = cycle * cycle; // squared gives inverse-square feel, 0 at far, 1 at close

  const flux = Math.round(20 + brightness * 480);

  // Spring physics for SB bar width
  // When moving: kick spring away from rest proportional to rate of change
  const rate = cycle - prevCycle;
  const isMoving = (state === 'MOVING_IN' || state === 'MOVING_OUT');
  if (isMoving) {
    sbSpringVel += rate * 280; // kick
  }
  // Spring force toward rest
  sbSpringVel += (SB_REST - sbSpringPos) * SB_SPRING;
  sbSpringVel *= SB_DAMP;
  sbSpringPos += sbSpringVel;
  prevCycle = cycle;

  const sbBarW = Math.max(10, Math.round(sbSpringPos));

  // Draw
  drawEuropa(europaX, europaY, europaR, brightness);
  drawSightLines(europaX, europaY);
  drawFixedPatch(europaX, europaY, brightness);
  drawDetector(brightness);

  // Readout panel
  ctx.fillStyle='rgba(14,18,32,0.92)';
  ctx.beginPath(); ctx.roundRect(22, 14, 290, 112, 8); ctx.fill();
  ctx.strokeStyle='rgba(100,120,180,0.18)'; ctx.lineWidth=0.5; ctx.stroke();

  // FLUX bar
  ctx.textAlign='left'; ctx.font='10px Arial';
  ctx.fillStyle='rgba(180,180,220,0.75)';
  ctx.fillText('FLUX  —  changes with distance', 34, 34);
  const bw = Math.max(6, (flux/500)*200);
  const rc = Math.round(100+brightness*155);
  const gc = Math.round(80+brightness*140);
  ctx.fillStyle=`rgba(${rc},${gc},70,0.9)`;
  ctx.beginPath(); ctx.roundRect(34, 40, bw, 12, 3); ctx.fill();
  ctx.fillStyle='rgba(200,200,230,0.65)';
  ctx.font='9px Arial';
  ctx.fillText(`${flux} Jy`, 34 + bw + 6, 51);

  // SURF_BRIGHT spring bar
  ctx.font='10px Arial';
  ctx.fillStyle='rgba(100,220,150,0.85)';
  ctx.fillText('SURF_BRIGHT  —  stays constant', 34, 72);

  // Track
  ctx.fillStyle='rgba(255,255,255,0.04)';
  ctx.beginPath(); ctx.roundRect(34, 79, 200, 12, 3); ctx.fill();

  // Rest position tick
  ctx.strokeStyle='rgba(100,220,150,0.3)';
  ctx.lineWidth=0.8; ctx.setLineDash([2,2]);
  ctx.beginPath(); ctx.moveTo(34+SB_REST, 77); ctx.lineTo(34+SB_REST, 93); ctx.stroke();
  ctx.setLineDash([]);

  // Spring bar — grows/shrinks, always anchored at left
  const sbAlpha = isMoving ? 0.6 : 0.9;
  ctx.fillStyle=`rgba(100,220,150,${sbAlpha})`;
  ctx.beginPath(); ctx.roundRect(34, 79, sbBarW, 12, 3); ctx.fill();

  ctx.fillStyle='rgba(100,220,150,0.55)';
  ctx.font='9px Arial';
  ctx.fillText('1.0 MJy/sr', 34 + Math.max(sbBarW, SB_REST) + 6, 90);

  // State label
  ctx.textAlign='center'; ctx.font='bold 10px Arial';
  if (state === 'PAUSE_CLOSE') {
    ctx.fillStyle='rgba(255,210,80,0.85)';
    ctx.fillText('CLOSE  —  bright', (europaX+hexCX)/2, 118);
  } else if (state === 'PAUSE_FAR') {
    ctx.fillStyle='rgba(120,170,255,0.85)';
    ctx.fillText('FAR  —  dim', (europaX+hexCX)/2, 118);
  }

  // Bottom explanation
  ctx.font='10px Arial';
  ctx.fillStyle='rgba(170,180,210,0.35)';
  ctx.fillText('Fixed angular patch · same solid angle always', W/2, 270);
  ctx.fillStyle='rgba(100,220,150,0.35)';
  ctx.fillText('Surface brightness = photons per steradian · conserved', W/2, 285);

  // Object labels
  ctx.fillStyle='#555577'; ctx.font='11px Arial';
  ctx.fillText('Europa', europaX, Math.min(252, europaY + europaR + 20));
  ctx.fillText('mirror segment', hexCX, hexCY + hexR*2.4 + 14);

  requestAnimationFrame(frame);
}
frame();
</script>

"""
with open('animation_03_surf_bright.html', 'w') as f:
    f.write(animation3)

print("Saved.")



Saved.


In [47]:
from IPython.display import HTML
HTML(animation3)

In [48]:
animation4 = """

<style>
  body { margin:0; background:#05080f; display:flex; align-items:center; justify-content:center; min-height:320px; }
  canvas { display:block; }
</style>
<canvas id="c" width="680" height="300"></canvas>
<script>
const canvas = document.getElementById('c');
const ctx = canvas.getContext('2d');
const W = 680, H = 300;
const EX = 155, EY = 150, ER = 110; // Europa
const DX = 530, DY = 150;           // Detector center

let t = 0;
// Which segment is "active" — cycles through all 19
let activeIdx = 0;
let dwellFrame = 0;
const DWELL = 60;

// Build JWST honeycomb — same math as previous animations
function buildJWST(cx, cy, R) {
  const cs = R*1.5, rs = R*Math.sqrt(3), st = R*Math.sqrt(3)/2;
  const colDefs = [
    { dx:-2*cs, ys:[-rs, 0, rs] },
    { dx:-cs,   ys:[-3*st,-st,st,3*st] },
    { dx:0,     ys:[-2*rs,-rs,0,rs,2*rs] },
    { dx:cs,    ys:[-3*st,-st,st,3*st] },
    { dx:2*cs,  ys:[-rs,0,rs] },
  ];
  const pos = [];
  colDefs.forEach(col => col.ys.forEach(dy => pos.push([cx+col.dx, cy+dy])));
  return pos;
}

// Detector hexes — R=13
const detHexes = buildJWST(DX, DY, 13);

// Europa overlay hexes — same shape, scaled to fit nicely on disk
// R=19 gives a 3-4-5-4-3 honeycomb that covers most of the disk
const europaHexes = buildJWST(EX, EY, 19);

function hexPath(cx, cy, r) {
  const pts = [];
  for (let i=0; i<6; i++) {
    const a = Math.PI/3*i;
    pts.push([cx+r*Math.cos(a), cy+r*Math.sin(a)]);
  }
  return pts;
}

function fillHex(cx, cy, r, fill, stroke, sw) {
  const pts = hexPath(cx, cy, r);
  ctx.beginPath();
  ctx.moveTo(pts[0][0], pts[0][1]);
  for (let i=1;i<6;i++) ctx.lineTo(pts[i][0], pts[i][1]);
  ctx.closePath();
  if (fill)   { ctx.fillStyle=fill;   ctx.fill(); }
  if (stroke) { ctx.strokeStyle=stroke; ctx.lineWidth=sw||0.5; ctx.stroke(); }
}

function drawEuropa() {
  const grad = ctx.createRadialGradient(EX-ER*0.28, EY-ER*0.28, ER*0.05, EX, EY, ER);
  grad.addColorStop(0,   '#e8f4ff');
  grad.addColorStop(0.3, '#c5dff5');
  grad.addColorStop(0.7, '#8ab8e8');
  grad.addColorStop(1,   '#3a6fa0');
  ctx.beginPath(); ctx.arc(EX, EY, ER, 0, Math.PI*2);
  ctx.fillStyle=grad; ctx.fill();

  // Cracks
  ctx.save();
  ctx.beginPath(); ctx.arc(EX, EY, ER, 0, Math.PI*2); ctx.clip();
  ctx.strokeStyle='rgba(110,68,48,0.5)'; ctx.lineWidth=1.2;
  [
    [[-0.42,-0.18],[0.08,0.38]],
    [[-0.18,0.42],[0.42,-0.08]],
    [[-0.32,-0.42],[0.32,-0.08]],
    [[0.18,-0.38],[0.48,0.22]],
    [[-0.48,0.12],[-0.08,0.48]],
  ].forEach(([a,b]) => {
    ctx.beginPath();
    ctx.moveTo(EX+ER*a[0],EY+ER*a[1]);
    ctx.lineTo(EX+ER*b[0],EY+ER*b[1]);
    ctx.stroke();
  });
  ctx.restore();

  // Terminator
  const shadow = ctx.createRadialGradient(EX+ER*0.4,EY,ER*0.1,EX,EY,ER);
  shadow.addColorStop(0.55,'rgba(0,8,24,0)');
  shadow.addColorStop(1,'rgba(0,8,24,0.38)');
  ctx.beginPath(); ctx.arc(EX,EY,ER,0,Math.PI*2);
  ctx.fillStyle=shadow; ctx.fill();

  // Atmosphere
  const atmo = ctx.createRadialGradient(EX,EY,ER*0.88,EX,EY,ER*1.12);
  atmo.addColorStop(0,'rgba(150,200,255,0.12)');
  atmo.addColorStop(1,'rgba(150,200,255,0)');
  ctx.beginPath(); ctx.arc(EX,EY,ER*1.12,0,Math.PI*2);
  ctx.fillStyle=atmo; ctx.fill();
}

function drawEuropaHexOverlay(activeIdx) {
  ctx.save();
  ctx.beginPath(); ctx.arc(EX,EY,ER,0,Math.PI*2); ctx.clip();
  europaHexes.forEach(([ hx, hy], i) => {
    const isActive = i === activeIdx;
    fillHex(hx, hy, 18.2,
      isActive ? 'rgba(255,220,80,0.32)' : 'rgba(255,255,255,0.04)',
      isActive ? 'rgba(255,220,80,0.9)' : 'rgba(255,255,255,0.25)',
      isActive ? 1.4 : 0.5
    );
  });
  ctx.restore();
}

// Draw a single traveling sine wave from src to dst
function drawTravelingWave(x0, y0, x1, y1, phaseOffset, alpha, lw) {
  const dx = x1-x0, dy = y1-y0;
  const len = Math.sqrt(dx*dx+dy*dy);
  const steps = Math.ceil(len/1.8);
  const wl = 34;
  const px = -dy/len, py = dx/len; // perpendicular

  ctx.beginPath();
  ctx.strokeStyle = `rgba(180,220,255,${alpha})`;
  ctx.lineWidth = lw;
  ctx.lineJoin = 'round';
  for (let i=0; i<=steps; i++) {
    const frac = i/steps;
    const bx = x0 + dx*frac;
    const by = y0 + dy*frac;
    const amp = 5 * Math.sin(Math.PI*frac); // taper
    const sine = amp * Math.sin(frac*len/wl*Math.PI*2 - t*4 + phaseOffset);
    const wx = bx + px*sine;
    const wy = by + py*sine;
    if (i===0) ctx.moveTo(wx,wy);
    else ctx.lineTo(wx,wy);
  }
  ctx.stroke();
}

function drawDetector(activeIdx) {
  detHexes.forEach(([hx,hy], i) => {
    const isActive = i === activeIdx;
    const localPulse = isActive ? (Math.sin(t*3)+1)/2 : 0;

    if (isActive) {
      // Halo
      const hg = ctx.createRadialGradient(hx,hy,0,hx,hy,22);
      hg.addColorStop(0,`rgba(255,220,80,${0.4+localPulse*0.5})`);
      hg.addColorStop(1,'rgba(255,220,80,0)');
      ctx.beginPath(); ctx.arc(hx,hy,22,0,Math.PI*2);
      ctx.fillStyle=hg; ctx.fill();
    }

    fillHex(hx, hy, 12.2,
      isActive
        ? `rgba(255,${Math.round(180+localPulse*75)},${Math.round(30+localPulse*180)},${0.5+localPulse*0.5})`
        : 'rgba(184,134,11,0.18)',
      isActive ? `rgba(255,210,50,0.9)` : 'rgba(184,134,11,0.55)',
      isActive ? 1.2 : 0.5
    );
  });
}

const bgStars = [
  [30,22,0.8,0.3],[620,28,0.7,0.4],[655,190,0.8,0.3],
  [18,265,0.7,0.3],[340,285,0.8,0.4],[660,265,0.7,0.3]
];

function frame() {
  ctx.clearRect(0,0,W,H);
  ctx.fillStyle='#05080f'; ctx.fillRect(0,0,W,H);

  bgStars.forEach(([x,y,r,a]) => {
    ctx.beginPath(); ctx.arc(x,y,r,0,Math.PI*2);
    ctx.fillStyle=`rgba(255,255,255,${a})`; ctx.fill();
  });

  t += 0.025;
  dwellFrame++;
  if (dwellFrame >= DWELL) {
    dwellFrame = 0;
    activeIdx = (activeIdx+1) % detHexes.length;
  }

  // Draw Europa base
  drawEuropa();
  // Draw hex overlay on Europa
  drawEuropaHexOverlay(activeIdx);

  // Draw all waves — all 19, active one bright, others dim
  detHexes.forEach(([dx2, dy2], i) => {
    const [ex2, ey2] = europaHexes[i];
    const isActive = i === activeIdx;
    drawTravelingWave(
      ex2, ey2, dx2, dy2,
      i * 0.38,
      isActive ? 0.9 : 0.08,
      isActive ? 1.3 : 0.4
    );
  });

  // Draw detector
  drawDetector(activeIdx);

  // Readout
  ctx.fillStyle='rgba(14,18,32,0.9)';
  ctx.beginPath(); ctx.roundRect(22,14,240,52,8); ctx.fill();
  ctx.strokeStyle='rgba(100,120,180,0.18)'; ctx.lineWidth=0.5; ctx.stroke();

  ctx.textAlign='left'; ctx.font='10px Arial';
  ctx.fillStyle='rgba(180,180,220,0.75)';
  ctx.fillText('Each hex cell → 1 mirror segment', 34, 32);
  ctx.fillStyle='rgba(100,220,150,0.85)';
  ctx.fillText('SURF_BRIGHT:', 34, 50);
  ctx.fillStyle='rgba(100,220,150,0.95)'; ctx.font='bold 10px Arial';
  ctx.fillText('1.0 MJy/sr  —  every cell, always', 118, 50);

  // Labels
  ctx.textAlign='center'; ctx.fillStyle='#555577'; ctx.font='11px Arial';
  ctx.fillText('Europa', EX, EY+ER+18);
  ctx.fillText('JWST NIRSpec', DX, DY+65);

  // Bottom
  ctx.font='10px Arial'; ctx.fillStyle='rgba(100,220,150,0.35)';
  ctx.fillText('Surface brightness is intrinsic to Europa — same from every patch', W/2, 290);

  requestAnimationFrame(frame);
}
frame();
</script>

"""
with open('animation_04_honeycomb_mapping.html', 'w') as f:
    f.write(animation4)

print("Saved.")

Saved.


In [49]:
from IPython.display import HTML
HTML(animation4)

### Journal Note
*I was so facinated with surface brightness measurments I kinda made a whole lesson on it. The animations turned out great but I can try to make them interactive (maybe add a slider to move Europa)*

*for some reason I'm not getting a frequency range in the data that would include water*


In [52]:
with fits.open(BytesIO(response.content)) as hdul:
    spec_data = hdul[1].data
    print(hdul.info())

    # Wavelength in microns (um)
    # This is our x-axis — WHERE in the spectrum we are
    wavelength = spec_data['WAVELENGTH']

    # Surface brightness — right measurement for extended objects
    # Europa is a disk, not a point source
    flux = spec_data['SURF_BRIGHT']
    error = spec_data['SB_ERROR']

    # Background — everything that isn't Europa
    # We'll subtract this to isolate Europa's true signal
    background = spec_data['BACKGROUND']

    # Background-subtracted signal — Europa's true spectrum
    # This is the scientific measurement we care about
    clean_flux = flux - background

    print("=== SPECTRAL DATA SUMMARY ===")
    print(f"Wavelength range: {wavelength.min():.3f} — {wavelength.max():.3f} microns")
    print(f"Number of spectral points: {len(wavelength)}")
    print(f"Wavelength resolution: {(wavelength.max()-wavelength.min())/len(wavelength)*1000:.3f} nm per point")
    print(f"\nWater vapor fingerprint at ~2.7 microns")
    print(f"In our range? {wavelength.min():.3f} < 2.7 < {wavelength.max():.3f}: "
          f"{bool(wavelength.min() < 2.7 < wavelength.max())}")
    print(f"\nSignal stats after background subtraction:")
    print(f"  Mean flux: {np.nanmean(clean_flux):.4f} MJy/sr")
    print(f"  Std dev:   {np.nanstd(clean_flux):.4f} MJy/sr")
    print(f"  Max flux:  {np.nanmax(clean_flux):.4f} MJy/sr")

Filename: <class '_io.BytesIO'>
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     257   ()      
  1  EXTRACT1D     1 BinTableHDU     70   3915R x 18C   ['D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'J', 'D', 'D', 'D', 'D', 'D', 'D']   
  2  ASDF          1 BinTableHDU     11   1R x 1C   [107276B]   
None
=== SPECTRAL DATA SUMMARY ===
Wavelength range: 0.970 — 1.890 microns
Number of spectral points: 3915
Wavelength resolution: 0.235 nm per point

Water vapor fingerprint at ~2.7 microns
In our range? 0.970 < 2.7 < 1.890: False

Signal stats after background subtraction:
  Mean flux: -9022.5468 MJy/sr
  Std dev:   305938.4759 MJy/sr
  Max flux:  110209.7788 MJy/sr


I did some googling and turns out NIRSpec splits its wavelength range across two detectors, NRS1 (shorter wavelengths) and NRS2 (longer wavelengths).

The file I fetched (nrs1_x1d.fits) only covers ~0.97–1.89 microns.
Water vapor absorption at 2.7 microns lives on NRS2.

Also NIRSpec has programable array of ~250,000 micro-shutters (the MSA) that physically
open and close to let specific targets' light through which is wild

In [56]:
with fits.open(BytesIO(response.content)) as hdul:
    header = hdul['PRIMARY'].header
    print(f"Telescope:     {header.get('TELESCOP', 'N/A')}")
    print(f"Instrument:    {header.get('INSTRUME', 'N/A')}")
    print(f"Detector:      {header.get('DETECTOR', 'N/A')}")

# CORRECTION: using NRS2 data

url = "https://mast.stsci.edu/api/v0.1/Download/file?uri=mast:JWST/product/jw01250006001_03101_00001_nrs2_x1d.fits"

print("Fetching Europa NRS2 spectral data from MAST...")
print("NRS2 covers ~1.7 - 3.2 microns — water vapor lives here\n")

start = time.time()
response2 = requests.get(url, stream=True)
elapsed = time.time() - start

print(f"Response status:  {response2.status_code}")
print(f"Content type:     {response2.headers.get('content-type')}")
print(f"Download time:    {elapsed:.2f} seconds")
print(f"File size:        {len(response2.content) / 1024:.1f} KB")

with fits.open(BytesIO(response2.content)) as hdul:
    header = hdul['PRIMARY'].header
    print(f"Telescope:     {header.get('TELESCOP', 'N/A')}")
    print(f"Instrument:    {header.get('INSTRUME', 'N/A')}")
    print(f"Detector:      {header.get('DETECTOR', 'N/A')}")
    hdul.info()
    spec_data     = hdul[1].data
    wavelength    = spec_data['WAVELENGTH']
    flux          = spec_data['SURF_BRIGHT']
    error         = spec_data['SB_ERROR']
    background    = spec_data['BACKGROUND']
    clean_flux    = flux - background

    print(f"\n=== NRS2 SPECTRAL DATA SUMMARY ===")
    print(f"Wavelength range:  {wavelength.min():.3f} — {wavelength.max():.3f} microns")
    print(f"Spectral points:   {len(wavelength)}")
    print(f"\nWater vapor fingerprint at ~2.7 microns")
    print(f"In our range? {wavelength.min():.3f} < 2.7 < {wavelength.max():.3f}: "
          f"{bool(wavelength.min() < 2.7 < wavelength.max())}")
    print(f"\nSignal stats after background subtraction:")
    print(f"  Mean:  {np.nanmean(clean_flux):.4f} MJy/sr")
    print(f"  Std:   {np.nanstd(clean_flux):.4f} MJy/sr")
    print(f"  Max:   {np.nanmax(clean_flux):.4f} MJy/sr")


Telescope:     JWST
Instrument:    NIRSPEC
Detector:      NRS2
Fetching Europa NRS2 spectral data from MAST...
NRS2 covers ~1.7 - 3.2 microns — water vapor lives here

Response status:  200
Content type:     application/octet-stream
Download time:    0.31 seconds
File size:        675.0 KB
Telescope:     JWST
Instrument:    NIRSPEC
Detector:      NRS2
Filename: <class '_io.BytesIO'>
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     257   ()      
  1  EXTRACT1D     1 BinTableHDU     70   3915R x 18C   [D, D, D, D, D, D, D, D, D, D, D, J, D, D, D, D, D, D]   
  2  ASDF          1 BinTableHDU     11   1R x 1C   [107277B]   

=== NRS2 SPECTRAL DATA SUMMARY ===
Wavelength range:  0.970 — 1.890 microns
Spectral points:   3915

Water vapor fingerprint at ~2.7 microns
In our range? 0.970 < 2.7 < 1.890: False

Signal stats after background subtraction:
  Mean:  92.4960 MJy/sr
  Std:   213.3968 MJy/sr
  Max:   8971.3213 MJy/sr


hmm I still get the same range but it did confirm that I'm using NRS2

In [57]:
from astroquery.mast import Observations

print("Searching MAST for JWST Europa NIRSpec observations...")
print("Program ID: 1250\n")

# Search by proposal ID
obs = Observations.query_criteria(
    proposal_id="1250",
    obs_collection="JWST",
    instrument_name="NIRSPEC/SLIT"
)

print(f"Found {len(obs)} observations")
print(obs['obs_id', 'instrument_name', 'filters', 't_exptime'][:10])

Searching MAST for JWST Europa NIRSpec observations...
Program ID: 1250



Found 0 observations
obs_id instrument_name filters t_exptime
------ --------------- ------- ---------
